# Iso-Scaling Significance Fraction Analysis

Analyze fraction of significant positive trends in iso-scaling results.

- Grouped by: dataset, paradigm, model
- Metrics: ZS (zero-shot) and CFT (finetuned)
- Significance: p < 0.05 (one-sided, positive direction)
- Counting: binary per (dataset, model, budget) - at least one iso point significant

In [ ]:
# %% imports and config
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

# config
POSITIVE_ONLY = False  # set False for two-sided
ALPHA = 0.01
FT_CONFIG_TAG = "FM-Full_A-None_AdaBN-F"
MODEL_ORDER = ["EEGNetv4", "ATCNet", "ShallowConvNet", "DeepConvNet"]

PARADIGM_MAP = {
    "Yang2025": "MI",
    "Lee2019_MI": "MI",
    "BNCI2014_001": "MI",
    "BI2015a": "P300",
    "Huebner2017": "P300",
    "Huebner2018": "P300",
    "Lee2019_SSVEP": "SSVEP",
    "Kalunga2016": "SSVEP",
    "MAMEM2": "SSVEP",
}

ISO_TRADE_OFF_POINTS = {
    "BI2015a": {
        "High": [(15, 1000), (20, 750), (30, 500)],
        "Low": [(10, 500), (20, 250), (25, 200)],
    },
    "Yang2025": {
        "High": [(15, 600), (20, 450), (30, 300)],
        "Low": [(10, 450), (15, 300), (30, 150)],
    },
    "Lee2019_SSVEP": {
        "High": [(20, 200), (25, 160), (40, 100)],
        "Low": [(10, 200), (20, 100), (40, 50)],
    },
    "Lee2019_MI": {
        "High": [(30, 200), (40, 150), (50, 120)],
        "Low": [(15, 200), (30, 100), (50, 60)],
    },
    "BNCI2014_001": {"High": [(6, 500), (8, 375)], "Low": [(3, 500), (5, 300)]},
    "Huebner2017": {
        "High": [(6, 12500), (10, 7500), (12, 6250)],
        "Low": [(3, 12500), (6, 6250), (10, 3750)],
    },
    "Huebner2018": {
        "High": [(6, 14000), (8, 10500), (12, 7000)],
        "Low": [(3, 14000), (6, 7000), (10, 4200)],
    },
    "Kalunga2016": {"High": [(8, 64), (12, 42)], "Low": [(4, 64), (8, 32)]},
    "MAMEM2": {"High": [(6, 100), (8, 75)], "Low": [(3, 100), (6, 50)]},
}

In [ ]:
# %% helper functions
def budget_for(dataset, nps, nt):
    points = ISO_TRADE_OFF_POINTS.get(dataset, {})
    for budget, coords in points.items():
        if (nps, nt) in coords:
            return budget
    return "Unknown"


def paired_ttest(base, cmp, positive_only=True):
    """Paired t-test. Returns (mean_delta, p_value)."""
    common = base.index.intersection(cmp.index)
    aligned = (
        base.loc[common]
        .rename("base")
        .to_frame()
        .join(cmp.loc[common].rename("cmp"), how="inner")
        .dropna()
    )
    if len(aligned) < 2:
        return np.nan, np.nan

    d = aligned["cmp"] - aligned["base"]
    mean_d = float(d.mean())

    if positive_only:
        # one-sided test: alternative = "greater" (cmp > base)
        p = float(
            stats.ttest_rel(
                aligned["cmp"], aligned["base"], alternative="greater"
            ).pvalue
        )
    else:
        # two-sided
        p = float(stats.ttest_rel(aligned["cmp"], aligned["base"]).pvalue)

    return mean_d, p


def load_iso_data(metric="finetuned"):
    """Load iso-ablation data. metric: 'finetuned' or 'zero_shot'"""
    path = Path("prepared_data/fig6_iso_ablation_complete.csv")
    df = pd.read_csv(path)
    df = df[df["config_tag"] == FT_CONFIG_TAG].copy()

    df["budget"] = [
        budget_for(ds, int(nps), int(nt))
        for ds, nps, nt in zip(
            df["dataset"], df["num_pretrain_subjects"], df["num_trials"]
        )
    ]
    df["paradigm"] = df["dataset"].map(PARADIGM_MAP)

    acc_col = f"{metric}_balanced_accuracy"
    df = df.rename(columns={acc_col: "accuracy"})

    return df[
        [
            "dataset",
            "paradigm",
            "model",
            "budget",
            "num_pretrain_subjects",
            "num_trials",
            "fold",
            "subject_id",
            "accuracy",
        ]
    ].copy()

In [ ]:
# %% compute significance for each (dataset, model, budget)
def compute_significance_flags(df, positive_only=True, alpha=0.05):
    """
    For each (dataset, model, budget), check if any iso point shows
    significant positive improvement vs the min-NPS baseline.

    Returns DataFrame with columns:
    - dataset, paradigm, model, budget
    - has_significant: bool (at least one comparison is significant)
    - best_delta: float (largest positive delta)
    - best_p: float (p-value of best comparison)
    """
    results = []

    for dataset in df["dataset"].unique():
        paradigm = PARADIGM_MAP.get(dataset, "Unknown")

        for model in MODEL_ORDER:
            for budget in ["Low", "High"]:
                pts = ISO_TRADE_OFF_POINTS.get(dataset, {}).get(budget, [])
                if not pts:
                    continue

                df_g = df[
                    (df["dataset"] == dataset)
                    & (df["model"] == model)
                    & (df["budget"] == budget)
                ].copy()

                if df_g.empty:
                    results.append(
                        {
                            "dataset": dataset,
                            "paradigm": paradigm,
                            "model": model,
                            "budget": budget,
                            "has_significant": np.nan,
                            "best_delta": np.nan,
                            "best_p": np.nan,
                            "n_comparisons": 0,
                        }
                    )
                    continue

                # baseline is min-NPS point
                base_nps, base_nt = sorted(pts, key=lambda t: t[0])[0]
                base = df_g[
                    (df_g["num_pretrain_subjects"] == base_nps)
                    & (df_g["num_trials"] == base_nt)
                ].set_index(["fold", "subject_id"])["accuracy"]

                # test each other point
                deltas, pvals = [], []
                for nps, nt in pts:
                    if (nps, nt) == (base_nps, base_nt):
                        continue  # skip baseline

                    cmp = df_g[
                        (df_g["num_pretrain_subjects"] == nps)
                        & (df_g["num_trials"] == nt)
                    ].set_index(["fold", "subject_id"])["accuracy"]

                    d, p = paired_ttest(base, cmp, positive_only=positive_only)
                    if np.isfinite(d):
                        deltas.append(d)
                        pvals.append(p)

                if not deltas:
                    results.append(
                        {
                            "dataset": dataset,
                            "paradigm": paradigm,
                            "model": model,
                            "budget": budget,
                            "has_significant": np.nan,
                            "best_delta": np.nan,
                            "best_p": np.nan,
                            "n_comparisons": 0,
                        }
                    )
                    continue

                # BH correction within this group
                pvals_adj = multipletests(pvals, method="fdr_bh")[1]

                # check for significance
                # two-sided: any significant effect (positive or negative) = evidence of trend
                # one-sided (positive_only): only positive effects count
                if positive_only:
                    sig_mask = (np.array(pvals_adj) < alpha) & (np.array(deltas) > 0)
                else:
                    sig_mask = np.array(pvals_adj) < alpha  # any direction
                has_sig = sig_mask.any()

                # best result (largest absolute effect if significant)
                if has_sig:
                    sig_idx = np.where(sig_mask)[0]
                    # pick the one with largest absolute delta
                    abs_deltas = np.abs(np.array(deltas)[sig_idx])
                    best_idx = sig_idx[np.argmax(abs_deltas)]
                    best_d, best_p = deltas[best_idx], pvals_adj[best_idx]
                else:
                    # report the largest absolute effect even if not significant
                    abs_deltas = np.abs(np.array(deltas))
                    best_idx = int(np.argmax(abs_deltas))
                    best_d = deltas[best_idx] if len(deltas) > 0 else np.nan
                    best_p = (
                        float(pvals_adj[best_idx]) if len(pvals_adj) > 0 else np.nan
                    )

                results.append(
                    {
                        "dataset": dataset,
                        "paradigm": paradigm,
                        "model": model,
                        "budget": budget,
                        "has_significant": has_sig,
                        "best_delta": best_d,
                        "best_p": best_p,
                        "n_comparisons": len(deltas),
                    }
                )

    return pd.DataFrame(results)

In [ ]:
# %% load data and compute significance
df_cft = load_iso_data(metric="finetuned")
df_zs = load_iso_data(metric="zero_shot")

print(f"CFT data: {len(df_cft)} rows")
print(f"ZS data: {len(df_zs)} rows")

sig_cft = compute_significance_flags(df_cft, positive_only=POSITIVE_ONLY, alpha=ALPHA)
sig_zs = compute_significance_flags(df_zs, positive_only=POSITIVE_ONLY, alpha=ALPHA)

sig_cft["metric"] = "CFT"
sig_zs["metric"] = "ZS"

sig_all = pd.concat([sig_cft, sig_zs], ignore_index=True)
print(f"\nTotal comparisons: {len(sig_all)}")

In [ ]:
# %% fraction significant by dataset
def fraction_significant(df_sig, group_cols):
    """Compute fraction of significant cases."""
    # drop rows with missing data
    df = df_sig.dropna(subset=["has_significant"]).copy()
    # convert boolean to int for proper aggregation
    df["has_significant"] = df["has_significant"].astype(int)

    agg = (
        df.groupby(group_cols)
        .agg(
            n_significant=("has_significant", "sum"),
            n_total=("has_significant", "count"),
        )
        .reset_index()
    )
    agg["fraction"] = agg["n_significant"] / agg["n_total"]
    return agg


print("=" * 60)
print("FRACTION SIGNIFICANT BY DATASET")
print("=" * 60)
frac_dataset = fraction_significant(sig_all, ["dataset", "metric"])
frac_dataset_pivot = frac_dataset.pivot(
    index="dataset", columns="metric", values="fraction"
)
frac_dataset_pivot = frac_dataset_pivot[["ZS", "CFT"]]  # reorder
print(frac_dataset_pivot.round(3).to_string())

In [ ]:
# %% fraction significant by paradigm
print("\n" + "=" * 60)
print("FRACTION SIGNIFICANT BY PARADIGM")
print("=" * 60)
frac_paradigm = fraction_significant(sig_all, ["paradigm", "metric"])
frac_paradigm_pivot = frac_paradigm.pivot(
    index="paradigm", columns="metric", values="fraction"
)
frac_paradigm_pivot = frac_paradigm_pivot[["ZS", "CFT"]]
print(frac_paradigm_pivot.round(3).to_string())

In [ ]:
# %% fraction significant by model
print("\n" + "=" * 60)
print("FRACTION SIGNIFICANT BY MODEL")
print("=" * 60)
frac_model = fraction_significant(sig_all, ["model", "metric"])
frac_model_pivot = frac_model.pivot(index="model", columns="metric", values="fraction")
frac_model_pivot = frac_model_pivot[["ZS", "CFT"]]
# reorder rows
frac_model_pivot = frac_model_pivot.reindex(MODEL_ORDER)
print(frac_model_pivot.round(3).to_string())

In [ ]:
# %% fraction significant by budget
print("\n" + "=" * 60)
print("FRACTION SIGNIFICANT BY BUDGET")
print("=" * 60)
frac_budget = fraction_significant(sig_all, ["budget", "metric"])
frac_budget_pivot = frac_budget.pivot(
    index="budget", columns="metric", values="fraction"
)
frac_budget_pivot = frac_budget_pivot[["ZS", "CFT"]]
print(frac_budget_pivot.round(3).to_string())

In [ ]:
# %% overall summary
print("\n" + "=" * 60)
print("OVERALL SUMMARY")
print("=" * 60)
frac_overall = fraction_significant(sig_all, ["metric"])
print(frac_overall.to_string(index=False))

In [ ]:
# %% detailed breakdown: paradigm x budget x metric
print("\n" + "=" * 60)
print("FRACTION SIGNIFICANT BY PARADIGM x BUDGET")
print("=" * 60)
frac_para_budget = fraction_significant(sig_all, ["paradigm", "budget", "metric"])
frac_para_budget_pivot = frac_para_budget.pivot(
    index=["paradigm", "budget"], columns="metric", values="fraction"
)[["ZS", "CFT"]]
print(frac_para_budget_pivot.round(3).to_string())

In [ ]:
# %% full detail table
print("\n" + "=" * 60)
print("FULL DETAIL: each (dataset, model, budget)")
print("=" * 60)

detail = sig_all.pivot(
    index=["dataset", "model", "budget"], columns="metric", values="has_significant"
)[["ZS", "CFT"]].reset_index()

# add paradigm
detail["paradigm"] = detail["dataset"].map(PARADIGM_MAP)
detail = detail[["paradigm", "dataset", "model", "budget", "ZS", "CFT"]]
detail = detail.sort_values(["paradigm", "dataset", "model", "budget"])

# show only rows where ZS and CFT differ
detail_diff = detail[detail["ZS"] != detail["CFT"]].copy()
print(f"\nCases where ZS and CFT differ ({len(detail_diff)} rows):")
print(detail_diff.to_string(index=False))

In [ ]:
# %% statistics when ZS has larger change than CFT
merged = sig_zs.merge(
    sig_cft,
    on=["dataset", "paradigm", "model", "budget"],
    suffixes=("_zs", "_cft"),
)

# filter: ZS has larger absolute delta than CFT
mask = merged["best_delta_zs"].abs() > merged["best_delta_cft"].abs()
subset = merged[mask].copy()

print("=" * 60)
print("STATISTICS: |delta| when ZS has larger change than CFT")
print("=" * 60)
print(f"N cases: {len(subset)}")
print(f"\nZS absolute delta:")
print(subset["best_delta_zs"].abs().describe().round(4).to_string())
print(f"\nCFT absolute delta (for comparison):")
print(subset["best_delta_cft"].abs().describe().round(4).to_string())

In [ ]:
# %% save results
output_path = Path("figure_data/iso_significance_fractions.csv")
sig_all.to_csv(output_path, index=False)
print(f"\nSaved detailed results to {output_path}")